# 🩺 DermoScan - Classificação de Lesões com Deep Learning

Este notebook organiza o fluxo completo do **DermoScan** para classificação de imagens dermatoscópicas com redes neurais convolucionais.

O roteiro está dividido em etapas curtas para facilitar a execução:

1. Preparação do ambiente.
2. Download e organização dos dados.
3. Treinamento e avaliação do modelo.
4. Uso do modelo treinado em uma interface interativa.

Ao longo do notebook, você encontrará blocos de markdown explicando o que acontece em cada etapa, por exemplo: **aqui você vai baixar o modelo treinado** ou **aqui você vai testar a imagem enviada pelo usuário**.

---

## Objetivo

O foco principal é apoiar a triagem de lesões de pele com uma abordagem voltada à redução de falsos negativos e à validação assistida pelo usuário.

## 1. Preparação do ambiente

Aqui você vai configurar o ambiente inicial e montar o acesso ao Colab Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Instalação das bibliotecas

Aqui você vai instalar as dependências necessárias para treino, avaliação e visualização.

In [ ]:
#instalação de lib
!pip install timm grad-cam kaggle -q
!pip install kagglehub -q

In [ ]:
# 2. Imports principais
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
import kagglehub

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import timm

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)

import torch.nn.functional as F

## 3. Download e organização dos dados

Aqui você vai baixar o dataset e preparar os caminhos das imagens e metadados.

In [ ]:
from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)

import torch.nn.functional as F# 1. Download do dataset
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Caminho para os arquivos do dataset:", path)

# 2. Carrega metadados originais
metadata = pd.read_csv(f'{path}/HAM10000_metadata.csv')

# 3. Localiza as imagens nas duas pastas part_1 e part_2
image_dir1 = f'{path}/HAM10000_images_part_1'
image_dir2 = f'{path}/HAM10000_images_part_2'

image_paths = {}
for folder in [image_dir1, image_dir2]:
    for file in os.listdir(folder):
        image_id = file.split('.')[0]
        image_paths[image_id] = os.path.join(folder, file)

# 4. Cria a coluna 'path'
metadata['path'] = metadata['image_id'].map(image_paths)

# 5. Cria a coluna numérica 'label' com base no diagnóstico 'dx'
metadata['label'] = metadata['dx'].astype('category').cat.codes
label_map = dict(enumerate(metadata['dx'].astype('category').cat.categories))
print("Mapeamento das classes numéricas:", label_map)
class_counts = metadata['dx'].value_counts()

plt.figure(figsize=(8,4))
sns.barplot(
    x=class_counts.index,
    y=class_counts.values
)
plt.title("Distribuição das Classes")
plt.show()

In [ ]:
# Treinamento, validação e teste com estratificação
train_df, temp_df = train_test_split(
    metadata,
    test_size=0.3,
    stratify=metadata['label'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['label'],
    random_state=42
)

print(f"Amostras - Treino: {len(train_df)} | Validação: {len(val_df)} | Teste: {len(test_df)}")

## 4. Separação dos dados

Aqui você vai dividir as amostras em treino, validação e teste com estratificação.

In [ ]:
IMG_SIZE = 300
BATCH_SIZE = 32


# DATA AUGMENTATION

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),

    transforms.RandomRotation(30),

    transforms.RandomAffine(
        degrees=15,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1)
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# VALIDAÇÃO E TESTE


val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# DATASET


class HAMDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row['path']).convert('RGB')
        label = torch.tensor(row['label'], dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label

# DATASETS


train_dataset = HAMDataset(train_df, train_transform)
val_dataset = HAMDataset(val_df, val_transform)
test_dataset = HAMDataset(test_df, val_transform)

# WEIGHTED RANDOM SAMPLER

class_counts = train_df['label'].value_counts().sort_index()

weights = 1.0 / class_counts

sample_weights = train_df['label'].map(weights)

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights.values),
    num_samples=len(sample_weights),
    replacement=True
)

# DATALOADERS


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Treino: {len(train_dataset)} imagens")
print(f"Validação: {len(val_dataset)} imagens")
print(f"Teste: {len(test_dataset)} imagens")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Treinando no dispositivo: {device}")

# 1. Configurando a ResNet18
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = resnet18.fc.in_features
resnet18.fc = nn.Linear(num_features, 7)
resnet18 = resnet18.to(device)

# EfficientNet-B3
efficientnet = timm.create_model(
    'efficientnet_b3',
    pretrained=True,
    num_classes=7
)

# Congela todas as camadas inicialmente
for param in efficientnet.parameters():
    param.requires_grad = False

# Libera apenas a camada classificadora
for param in efficientnet.classifier.parameters():
    param.requires_grad = True

efficientnet = efficientnet.to(device)

In [ ]:
# Pesos das classes
classes_treino = train_df['label'].values

pesos_treino = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(classes_treino),
    y=classes_treino
)

class_weights_tensor = torch.FloatTensor(
    pesos_treino
).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


def train_model(
    model,
    train_loader,
    val_loader,
    model_name='model',
    epochs=20
):

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad,
               model.parameters()),
        lr=1e-4
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=2
    )

    best_val_loss = float('inf')

    patience = 5
    counter = 0

    train_losses = []
    val_losses = []

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0

        for images, labels in tqdm(
            train_loader,
            desc=f"{model_name} - Epoch {epoch+1}"
        ):

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)


  # VALIDAÇÃO


        model.eval()

        val_running_loss = 0.0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                loss = criterion(outputs, labels)

                val_running_loss += loss.item()

        val_loss = val_running_loss / len(val_loader)

        scheduler.step(val_loss)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(
            f'[{model_name}] Epoch {epoch+1} | '
            f'Train: {train_loss:.4f} | '
            f'Val: {val_loss:.4f}'
        )


# SALVA MELHOR MODELO


        if val_loss < best_val_loss:

            best_val_loss = val_loss

            counter = 0

            torch.save(
                model.state_dict(),
                f'best_{model_name}.pth'
            )

            print(
                f'=> Melhor modelo '
                f'{model_name} salvo!'
            )

        else:
            counter += 1


# EARLY STOPPING


        if counter >= patience:

            print(
                f"Early stopping ativado "
                f"para {model_name}"
            )

            break

    return train_losses, val_losses

#Não execute o bloco de Treino, pule para baixar o modelo já treinado

In [ ]:
# TREINAMENTO EFFICIENTNET - FASE 1
# Apenas classificador treinável

train_losses_eff_stage1, val_losses_eff_stage1 = train_model(
    model=efficientnet,
    train_loader=train_loader,
    val_loader=val_loader,
    model_name='efficientnet_stage1',
    epochs=10
)

# FINE TUNING - FASE 2

for param in efficientnet.parameters():
    param.requires_grad = True

print("Fine-Tuning iniciado...")

train_losses_eff_stage2, val_losses_eff_stage2 = train_model(
    model=efficientnet,
    train_loader=train_loader,
    val_loader=val_loader,
    model_name='efficientnet_finetuned',
    epochs=10
)

print("Treinamento concluído!")

#Rode este bloco ao baixar o arquivo 'best_efficientnet_finetuned.pth', ele é o modelo já treinado

In [ ]:
efficientnet.load_state_dict(
    torch.load(
        'best_efficientnet_finetuned.pth',
        map_location=device
    )
)

efficientnet.eval()

In [ ]:
efficientnet.load_state_dict(
    torch.load(
        'best_efficientnet_finetuned.pth',
        map_location=device
    )
)

efficientnet.eval()

print("Melhor modelo carregado com sucesso!")

## 7. Métricas e relatório final

Aqui você vai conferir os indicadores globais do modelo treinado.

In [ ]:
def encontrar_limiares_otimos(probs, labels, label_map):
    """
    Calcula o limiar ideal para cada classe usando o Índice de Youden.
    """

    limiares_otimos = {}

    print("--- LIMITARES ÓTIMOS CALCULADOS POR CLASSE ---")

    for idx, classe in label_map.items():

        y_true_binario = (labels == idx).astype(int)
        y_prob_classe = probs[:, idx]

        fpr, tpr, thresholds = roc_curve(
            y_true_binario,
            y_prob_classe
        )

        youden_index = tpr - fpr

        idx_otimo = np.argmax(youden_index)

        limiar_otimo = thresholds[idx_otimo]

        if limiar_otimo > 1:
            limiar_otimo = 1.0

        if limiar_otimo < 0:
            limiar_otimo = 0.0

        limiares_otimos[classe] = limiar_otimo

        print(
            f"Lesão: {classe.upper().ljust(6)} "
            f"| Limiar Ideal: {limiar_otimo:.4f}"
        )

    return limiares_otimos


def evaluate_model_with_thresholds(
    model,
    weight_path,
    test_loader,
    label_map
):

    model.load_state_dict(
        torch.load(
            weight_path,
            map_location=device
        )
    )

    model.eval()

    preds_originais = []
    probs_lista = []
    labels_list = []

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)

            outputs = model(images)

            probabilities = torch.softmax(
                outputs,
                dim=1
            )

            _, predicted = torch.max(
                outputs,
                1
            )

            preds_originais.extend(
                predicted.cpu().numpy()
            )

            probs_lista.extend(
                probabilities.cpu().numpy()
            )

            labels_list.extend(
                labels.numpy()
            )

    probs_array = np.array(probs_lista)
    labels_array = np.array(labels_list)

    limiares = encontrar_limiares_otimos(
        probs_array,
        labels_array,
        label_map
    )

    preds_ajustadas = []

    for i in range(len(probs_array)):

        prob_amostra = probs_array[i]

        classes_acima_do_limiar = []

        for idx, classe in label_map.items():

            if prob_amostra[idx] >= limiares[classe]:
                classes_acima_do_limiar.append(idx)

        if len(classes_acima_do_limiar) == 1:

            preds_ajustadas.append(
                classes_acima_do_limiar[0]
            )

        elif len(classes_acima_do_limiar) > 1:

            valores_prob = [
                prob_amostra[c]
                for c in classes_acima_do_limiar
            ]

            preds_ajustadas.append(
                classes_acima_do_limiar[
                    np.argmax(valores_prob)
                ]
            )

        else:

            preds_ajustadas.append(
                np.argmax(prob_amostra)
            )

    return (
        np.array(preds_ajustadas),
        probs_array,
        labels_array,
        limiares
    )


# AVALIAÇÃO DO MELHOR MODELO

preds_eff, probs_eff, labels_eff, dicionario_limiares = (
    evaluate_model_with_thresholds(
        efficientnet,
        'best_efficientnet_finetuned.pth',
        test_loader,
        label_map
    )
)

print(
    "\n--- RELATÓRIO EFFICIENTNET-B3 "
    "(COM LIMIARES CUSTOMIZADOS) ---"
)

print(
    classification_report(
        labels_eff,
        preds_eff,
        target_names=list(label_map.values())
    )
)

# MÉTRICAS GERAIS

acc = accuracy_score(
    labels_eff,
    preds_eff
)

f1_macro = f1_score(
    labels_eff,
    preds_eff,
    average='macro'
)

auc_macro = roc_auc_score(
    pd.get_dummies(labels_eff),
    probs_eff,
    multi_class='ovr',
    average='macro'
)

print(f"\nAccuracy : {acc:.4f}")
print(f"F1 Macro : {f1_macro:.4f}")
print(f"AUC Macro: {auc_macro:.4f}")

# MATRIZ DE CONFUSÃO

cm = confusion_matrix(
    labels_eff,
    preds_eff
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=list(label_map.values()),
    yticklabels=list(label_map.values())
)

plt.title(
    'Matriz de Confusão - EfficientNet B3'
)

plt.xlabel('Predito')
plt.ylabel('Real')

plt.tight_layout()
plt.show()

In [ ]:
# MÉTRICAS FINAIS

acc_eff = accuracy_score(
    labels_eff,
    preds_eff
)

f1_eff = f1_score(
    labels_eff,
    preds_eff,
    average='macro'
)

auc_eff_macro = roc_auc_score(
    pd.get_dummies(labels_eff),
    probs_eff,
    average='macro',
    multi_class='ovr'
)

print(f"Acurácia : {acc_eff:.4f}")
print(f"F1 Macro : {f1_eff:.4f}")
print(f"AUC Macro: {auc_eff_macro:.4f}")

# TABELA DE RESULTADOS

comparison = pd.DataFrame({
    'Métrica': [
        'Acurácia (Accuracy)',
        'F1-Score (Macro)',
        'AUC-ROC Média (Macro)'
    ],
    'EfficientNet-B3': [
        acc_eff,
        f1_eff,
        auc_eff_macro
    ]
})

display(comparison)

# GRÁFICO DE PERFORMANCE

ax = comparison.plot(
    x='Métrica',
    kind='bar',
    figsize=(10, 6),
    color='#1f77b4',
    legend=False
)

plt.title(
    'Performance Geral do Modelo EfficientNet-B3'
)

plt.ylabel('Score')
plt.ylim(0, 1.1)

plt.xticks(rotation=0)

plt.grid(
    axis='y',
    linestyle='--',
    alpha=0.7
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.3f',
        padding=3
    )

plt.tight_layout()
plt.show()

In [ ]:
# MATRIZ DE CONFUSÃO

nomes_extenso = [
    'Queratose Actínica',
    'Carcinoma Basocelular',
    'Cerne Benigno (BKL)',
    'Dermatofibroma',
    'Melanoma',
    'Nevo Melanocítico',
    'Lesão Vascular'
]

cm = confusion_matrix(
    labels_eff,
    preds_eff
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=nomes_extenso,
    yticklabels=nomes_extenso
)

plt.title(
    'Matriz de Confusão - EfficientNet-B3',
    fontsize=14,
    pad=20
)

plt.xlabel(
    'Classe Predita',
    fontsize=12
)

plt.ylabel(
    'Classe Real',
    fontsize=12
)

plt.xticks(
    rotation=45,
    ha='right'
)

plt.yticks(
    rotation=0
)

plt.tight_layout()

plt.show()

# MATRIZ DE CONFUSÃO NORMALIZADA

cm_norm = confusion_matrix(
    labels_eff,
    preds_eff,
    normalize='true'
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm_norm,
    annot=True,
    fmt='.2f',
    cmap='Greens',
    xticklabels=nomes_extenso,
    yticklabels=nomes_extenso
)

plt.title(
    'Matriz de Confusão Normalizada (%)',
    fontsize=14,
    pad=20
)

plt.xlabel(
    'Classe Predita',
    fontsize=12
)

plt.ylabel(
    'Classe Real',
    fontsize=12
)

plt.xticks(
    rotation=45,
    ha='right'
)

plt.yticks(
    rotation=0
)

plt.tight_layout()

plt.show()

In [ ]:
efficientnet.eval()

caminhos_arquivos = []
diagnosticos_reais = []
diagnosticos_preditos_originais = []
diagnosticos_preditos_com_limiar = []

# Probabilidades por classe
probabilidades_por_classe = {
    classe: []
    for classe in label_map.values()
}

with torch.no_grad():

    for idx in tqdm(
        range(len(test_df)),
        desc="Analisando conjunto de teste"
    ):

        row = test_df.iloc[idx]

        caminho_img = row['path']
        label_real = row['label']

        imagem_pura = Image.open(
            caminho_img
        ).convert('RGB')

        imagem_tensor = (
            val_transform(imagem_pura)
            .unsqueeze(0)
            .to(device)
        )

        saida = efficientnet(imagem_tensor)

        probs = torch.softmax(
            saida,
            dim=1
        ).cpu().numpy()[0]

        _, predicted = torch.max(
            saida,
            1
        )

        caminhos_arquivos.append(
            caminho_img
        )

        diagnosticos_reais.append(
            label_map[int(label_real)]
        )

        diagnosticos_preditos_originais.append(
            label_map[predicted.item()]
        )

        # Salva probabilidades
        for i, classe in label_map.items():

            probabilidades_por_classe[
                classe
            ].append(
                probs[i]
            )

        # Aplicação dos limiares
        classes_acima_do_limiar = []

        for i, classe in label_map.items():

            if probs[i] >= dicionario_limiares[classe]:

                classes_acima_do_limiar.append(i)

        if len(classes_acima_do_limiar) == 1:

            nova_pred = classes_acima_do_limiar[0]

        elif len(classes_acima_do_limiar) > 1:

            valores_prob = [
                probs[c]
                for c in classes_acima_do_limiar
            ]

            nova_pred = classes_acima_do_limiar[
                np.argmax(valores_prob)
            ]

        else:

            nova_pred = np.argmax(probs)

        diagnosticos_preditos_com_limiar.append(
            label_map[nova_pred]
        )

# DATAFRAME FINAL

dados_resultado = {

    'Caminho_Imagem':
        caminhos_arquivos,

    'Diagnostico_Real':
        diagnosticos_reais,

    'Predicao_Original':
        diagnosticos_preditos_originais,

    'Predicao_Com_Limiar':
        diagnosticos_preditos_com_limiar
}

for classe, lista_probs in probabilidades_por_classe.items():

    dados_resultado[
        f'Probabilidade_{classe.upper()}'
    ] = lista_probs

resultados_teste = pd.DataFrame(
    dados_resultado
)

resultados_teste[
    'Acertou_Original'
] = (
    resultados_teste['Diagnostico_Real']
    ==
    resultados_teste['Predicao_Original']
)

resultados_teste[
    'Acertou_Com_Limiar'
] = (
    resultados_teste['Diagnostico_Real']
    ==
    resultados_teste['Predicao_Com_Limiar']
)

# SALVAR CSV

resultados_teste.to_csv(
    '/content/drive/MyDrive/modelos/resultados_teste_efficientnet.csv',
    index=False
)

print(
    "✅ Resultados salvos em:"
)

print(
    "/content/drive/MyDrive/modelos/resultados_teste_efficientnet.csv"
)

# VISUALIZAÇÃO

display(
    resultados_teste.head()
)

print(
    f"\nTotal de imagens avaliadas: {len(resultados_teste)}"
)

print(
    f"Acertos Originais: "
    f"{resultados_teste['Acertou_Original'].mean()*100:.2f}%"
)

print(
    f"Acertos com Limiar: "
    f"{resultados_teste['Acertou_Com_Limiar'].mean()*100:.2f}%"
)

In [ ]:
# GRAD-CAM - ANÁLISE DE ERRO

model = efficientnet
model.eval()

target_layers = [model.conv_head]

cam = GradCAM(
    model=model,
    target_layers=target_layers
)

# ESCOLHE AUTOMATICAMENTE UM ERRO

erros = resultados_teste[
    resultados_teste['Acertou_Com_Limiar'] == False
]

if len(erros) == 0:
    print("✅ Nenhum erro encontrado.")
else:

    exemplo = erros.iloc[0]

    caminho_imagem_erro = exemplo['Caminho_Imagem']

    print("Imagem analisada:")
    print(caminho_imagem_erro)

    print(
        f"Real: {exemplo['Diagnostico_Real']}"
    )

    print(
        f"Predito: {exemplo['Predicao_Com_Limiar']}"
    )

    # CARREGAMENTO DA IMAGEM

    imagem_pil = Image.open(
        caminho_imagem_erro
    ).convert('RGB')

    input_tensor = (
        val_transform(imagem_pil)
        .unsqueeze(0)
        .to(device)
    )

    # INFERÊNCIA

    outputs = model(input_tensor)

    probs = torch.softmax(
        outputs,
        dim=1
    ).cpu().detach().numpy()[0]

    predicted_class = outputs.argmax(
        dim=1
    ).item()

    targets = [
        ClassifierOutputTarget(
            predicted_class
        )
    ]

    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=targets
    )

    grayscale_cam = grayscale_cam[0, :]

    # VISUALIZAÇÃO

    img_render = imagem_pil.resize(
        (IMG_SIZE, IMG_SIZE)
    )

    img_render = (
        np.float32(img_render)
        / 255.0
    )

    visualization = show_cam_on_image(
        img_render,
        grayscale_cam,
        use_rgb=True
    )

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)

    plt.imshow(img_render)

    plt.title(
        f"Imagem Original\n"
        f"Real: {exemplo['Diagnostico_Real']}"
    )

    plt.axis('off')

    plt.subplot(1, 2, 2)

    plt.imshow(visualization)

    plt.title(
        f"Grad-CAM\n"
        f"Predito: {label_map[predicted_class]}"
    )

    plt.axis('off')

    plt.tight_layout()

    plt.show()

    # PROBABILIDADES

    print("\nProbabilidades:")

    for idx, classe in label_map.items():

        print(
            f"{classe.upper():<8} "
            f"{probs[idx]*100:.2f}%"
        )

In [ ]:
import random

def testar_imagem_do_dataset_com_limiares(
    df_teste,
    modelo,
    mapeamento_classes,
    dicionario_limiares,
    device
):

    modelo.eval()

    # ESCOLHE IMAGEM ALEATÓRIA

    linha_aleatoria = df_teste.sample(
        n=1
    ).iloc[0]

    caminho_img = linha_aleatoria['path']

    classe_real_idx = int(
        linha_aleatoria['label']
    )

    classe_real_nome = (
        mapeamento_classes[
            classe_real_idx
        ]
    )

    # TRANSFORMAÇÃO

    transformacao = transforms.Compose([
        transforms.Resize(
            (IMG_SIZE, IMG_SIZE)
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ])

    imagem_pura = Image.open(
        caminho_img
    ).convert('RGB')

    imagem_tensor = (
        transformacao(imagem_pura)
        .unsqueeze(0)
        .to(device)
    )

    # INFERÊNCIA

    with torch.no_grad():

        saida = modelo(
            imagem_tensor
        )

        probabilidades = torch.softmax(
            saida,
            dim=1
        ).cpu().numpy()[0]

        indice_predito_original = np.argmax(
            probabilidades
        )

    classe_predita_original = (
        mapeamento_classes[
            indice_predito_original
        ]
    )

    confianca_original = (
        probabilidades[
            indice_predito_original
        ] * 100
    )

    # LIMIARES

    classes_acima_do_limiar = []

    for idx, classe in mapeamento_classes.items():

        if (
            probabilidades[idx]
            >=
            dicionario_limiares[classe]
        ):

            classes_acima_do_limiar.append(
                idx
            )

    if len(classes_acima_do_limiar) == 1:

        idx_final = classes_acima_do_limiar[0]

    elif len(classes_acima_do_limiar) > 1:

        valores_prob = [
            probabilidades[c]
            for c in classes_acima_do_limiar
        ]

        idx_final = (
            classes_acima_do_limiar[
                np.argmax(valores_prob)
            ]
        )

    else:

        idx_final = indice_predito_original

    classe_predita_limiar = (
        mapeamento_classes[idx_final]
    )

    confianca_limiar = (
        probabilidades[idx_final]
        * 100
    )

    # VISUALIZAÇÃO

    plt.figure(figsize=(6, 6))

    plt.imshow(imagem_pura)

    cor_titulo = (
        'green'
        if classe_predita_limiar
        ==
        classe_real_nome
        else 'red'
    )

    texto_titulo = (
        f"Real: {classe_real_nome.upper()}\n"
        f"Predição Original: {classe_predita_original} ({confianca_original:.1f}%)\n"
        f"Predição Final: {classe_predita_limiar} ({confianca_limiar:.1f}%)"
    )

    plt.title(
        texto_titulo,
        color=cor_titulo,
        fontsize=11,
        loc='left'
    )

    plt.axis('off')

    plt.tight_layout()

    plt.show()

    # PROBABILIDADES ORDENADAS

    print("\nRanking das Classes")

    ranking = []

    for idx, classe in mapeamento_classes.items():

        ranking.append(
            (
                classe,
                probabilidades[idx]
            )
        )

    ranking = sorted(
        ranking,
        key=lambda x: x[1],
        reverse=True
    )

    for classe, prob in ranking:

        status = ""

        if (
            prob
            >=
            dicionario_limiares[classe]
        ):
            status = "✅"

        print(
            f"{classe.upper():<8} "
            f"{prob*100:6.2f}% "
            f"{status}"
        )

# EXECUÇÃO

testar_imagem_do_dataset_com_limiares(
    test_df,
    efficientnet,
    label_map,
    dicionario_limiares,
    device
)

## 8. Comparativo de limiares

Aqui você vai visualizar os limiares ótimos calculados para cada classe.

In [ ]:
# Exibe um relatório visual detalhado dos limiares que foram calculados no bloco de avaliação
print("="*50)
print("   LIMIARES ÓTIMOS DE DECISÃO (CURVA ROC / YOUDEN)   ")
print("="*50)
print("Cada classe abaixo recebeu um ponto de corte personalizado.")
print("Se a probabilidade da IA superar esse valor, a lesão é considerada ativada.\n")

for classe, limiar in dicionario_limiares.items():
    print(f"Lesão: {classe.upper().ljust(6)} | Limiar de ativação: {limiar * 100:6.2f}%")
print("="*50)

In [ ]:
preds_eff, probs_eff, labels_eff, dicionario_limiares = (
    evaluate_model_with_thresholds(
        efficientnet,
        'best_efficientnet_finetuned.pth',
        test_loader,
        label_map
    )
)

print(
    "\n=== RELATÓRIO EFFICIENTNET-B3 "
    "(LIMIARES ÓTIMOS ROC/YOUDEN) ===\n"
)

print(
    classification_report(
        labels_eff,
        preds_eff,
        target_names=list(label_map.values()),
        digits=4
    )
)

# MÉTRICAS GERAIS

acc_eff = accuracy_score(
    labels_eff,
    preds_eff
)

f1_eff = f1_score(
    labels_eff,
    preds_eff,
    average='macro'
)

auc_eff = roc_auc_score(
    pd.get_dummies(labels_eff),
    probs_eff,
    average='macro',
    multi_class='ovr'
)

print(f"\nAccuracy : {acc_eff:.4f}")
print(f"F1 Macro : {f1_eff:.4f}")
print(f"AUC Macro: {auc_eff:.4f}")

In [ ]:
# GRÁFICO DOS LIMIARES

df_limiares = pd.DataFrame({
    'Lesão': [
        classe.upper()
        for classe in dicionario_limiares.keys()
    ],
    'Limiar de Decisão': list(
        dicionario_limiares.values()
    )
})

# Ordena do maior para o menor
df_limiares = df_limiares.sort_values(
    'Limiar de Decisão',
    ascending=False
)

plt.figure(figsize=(10, 5))

ax = sns.barplot(
    data=df_limiares,
    x='Lesão',
    y='Limiar de Decisão',
    hue='Lesão',
    legend=False,
    palette='Oranges_r'
)

plt.title(
    'Limiares Ótimos por Tipo de Lesão',
    fontsize=14,
    pad=15
)

plt.ylabel(
    'Limiar (Threshold)',
    fontsize=12
)

plt.xlabel(
    'Classe',
    fontsize=12
)

plt.ylim(0, 1.1)

for container in ax.containers:

    ax.bar_label(
        container,
        fmt='%.3f',
        padding=3
    )

plt.grid(
    axis='y',
    linestyle='--',
    alpha=0.4
)

plt.tight_layout()

plt.show()

# TABELA ORDENADA

display(df_limiares)

## 9. Teste com limiar

Aqui você vai testar o modelo em uma imagem do conjunto de teste usando os limiares ajustados.

In [ ]:
import random

def testar_imagem_do_dataset_com_limiares(
    df_teste,
    modelo,
    mapeamento_classes,
    dicionario_limiares,
    device
):

    modelo.eval()

    # Seleciona uma imagem aleatória
    linha_aleatoria = df_teste.sample(
        n=1,
        random_state=None
    ).iloc[0]

    caminho_img = linha_aleatoria['path']

    classe_real_idx = int(
        linha_aleatoria['label']
    )

    classe_real_nome = (
        mapeamento_classes[
            classe_real_idx
        ]
    )

    # Usa o mesmo tamanho do treinamento
    transformacao = transforms.Compose([
        transforms.Resize(
            (IMG_SIZE, IMG_SIZE)
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ])

    imagem_pura = Image.open(
        caminho_img
    ).convert('RGB')

    imagem_tensor = (
        transformacao(imagem_pura)
        .unsqueeze(0)
        .to(device)
    )

    # Inferência
    with torch.no_grad():

        saida = modelo(imagem_tensor)

        probabilidades = torch.softmax(
            saida,
            dim=1
        ).cpu().numpy()[0]

        indice_predito_original = np.argmax(
            probabilidades
        )

    classe_predita_original = (
        mapeamento_classes[
            indice_predito_original
        ]
    )

    confianca_original = (
        probabilidades[
            indice_predito_original
        ] * 100
    )

    # Aplicação dos limiares
    classes_acima_do_limiar = []

    for idx, classe in mapeamento_classes.items():

        if (
            probabilidades[idx]
            >=
            dicionario_limiares[classe]
        ):

            classes_acima_do_limiar.append(idx)

    if len(classes_acima_do_limiar) == 1:

        idx_final = classes_acima_do_limiar[0]

    elif len(classes_acima_do_limiar) > 1:

        valores_prob = [
            probabilidades[c]
            for c in classes_acima_do_limiar
        ]

        idx_final = (
            classes_acima_do_limiar[
                np.argmax(valores_prob)
            ]
        )

    else:

        idx_final = indice_predito_original

    classe_predita_limiar = (
        mapeamento_classes[idx_final]
    )

    confianca_limiar = (
        probabilidades[idx_final]
        * 100
    )

    # Visualização
    plt.figure(figsize=(6, 6))

    plt.imshow(imagem_pura)

    cor_titulo = (
        'green'
        if classe_predita_limiar
        ==
        classe_real_nome
        else 'red'
    )

    texto_titulo = (
        f"Real: {classe_real_nome.upper()}\n"
        f"Original: {classe_predita_original.upper()} ({confianca_original:.1f}%)\n"
        f"Final: {classe_predita_limiar.upper()} ({confianca_limiar:.1f}%)"
    )

    plt.title(
        texto_titulo,
        color=cor_titulo,
        fontsize=11,
        loc='left',
        pad=10
    )

    plt.axis('off')

    plt.tight_layout()

    plt.show()

    # Ranking completo das classes

    print("\n--- Ranking das Probabilidades ---")

    ranking = []

    for idx, classe in mapeamento_classes.items():

        ranking.append(
            (
                classe,
                probabilidades[idx]
            )
        )

    ranking = sorted(
        ranking,
        key=lambda x: x[1],
        reverse=True
    )

    for classe, prob in ranking:

        status = ""

        if (
            prob
            >=
            dicionario_limiares[classe]
        ):
            status = "✅"

        print(
            f"{classe.upper():<8} "
            f"{prob*100:6.2f}% "
            f"{status}"
        )


# EXECUÇÃO

testar_imagem_do_dataset_com_limiares(
    test_df,
    efficientnet,
    label_map,
    dicionario_limiares,
    device
)

In [ ]:
import io
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

HITL_ROOT = Path.cwd() / 'hitl_feedback'
HITL_IMAGES_DIR = HITL_ROOT / 'images'
HITL_LOG_PATH = HITL_ROOT / 'validated_samples.csv'
HITL_STATE_PATH = HITL_ROOT / 'state.json'
HITL_RETRAIN_EVERY = 100
HITL_RETRAIN_EPOCHS = 1
HITL_RETRAIN_BATCH_SIZE = 16
HITL_RETRAIN_LR = 1e-5

HITL_ROOT.mkdir(parents=True, exist_ok=True)
HITL_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

class_names = list(label_map.values())
class_to_idx = {classe: idx for idx, classe in label_map.items()}
status_feedback = None

try:
    feedback_log = pd.read_csv(HITL_LOG_PATH)
except FileNotFoundError:
    feedback_log = pd.DataFrame(columns=[
        'record_id',
        'created_at',
        'image_path',
        'predicted_label',
        'predicted_confidence',
        'raw_prediction',
        'raw_prediction_confidence',
        'feedback',
        'correct_label',
        'notes',
    ])

try:
    with open(HITL_STATE_PATH, 'r', encoding='utf-8') as state_file:
        hitl_state = json.load(state_file)
except FileNotFoundError:
    hitl_state = {'last_retrain_count': 0}


def _save_state():
    with open(HITL_STATE_PATH, 'w', encoding='utf-8') as state_file:
        json.dump(hitl_state, state_file, ensure_ascii=False, indent=2)


def _uploaded_content(upload_widget):
    value = upload_widget.value
    if isinstance(value, dict):
        return list(value.values())[0]['content']
    return list(value)[0]['content']


def _hitl_transform():
    if 'train_transform' in globals():
        return train_transform
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


class HITLFeedbackDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = Image.open(row['image_path']).convert('RGB')
        image_tensor = self.transform(image)
        label_tensor = torch.tensor(class_to_idx[row['correct_label']], dtype=torch.long)
        return image_tensor, label_tensor


def _save_feedback(image_pil, predicted_label, predicted_confidence, raw_prediction, raw_prediction_confidence, feedback, correct_label, notes=''):
    record_id = uuid.uuid4().hex
    image_path = HITL_IMAGES_DIR / f'{record_id}.png'
    image_pil.save(image_path)

    global feedback_log
    new_row = pd.DataFrame([{
        'record_id': record_id,
        'created_at': datetime.now(timezone.utc).isoformat(),
        'image_path': str(image_path),
        'predicted_label': predicted_label,
        'predicted_confidence': float(predicted_confidence),
        'raw_prediction': raw_prediction,
        'raw_prediction_confidence': float(raw_prediction_confidence),
        'feedback': feedback,
        'correct_label': correct_label,
        'notes': notes,
    }])
    feedback_log = pd.concat([feedback_log, new_row], ignore_index=True)
    feedback_log.to_csv(HITL_LOG_PATH, index=False)


def _maybe_retrain():
    total_validated = len(feedback_log)
    validated_since_last = total_validated - int(hitl_state.get('last_retrain_count', 0))

    if validated_since_last < HITL_RETRAIN_EVERY:
        remaining = HITL_RETRAIN_EVERY - validated_since_last
        if status_feedback is not None:
            status_feedback.value = f'<b>Status:</b> feedback salvo. Faltam {remaining} validações para o próximo re-treino.'
        return

    labeled_df = feedback_log[feedback_log['correct_label'].isin(class_names)].copy()
    if len(labeled_df) < 2:
        if status_feedback is not None:
            status_feedback.value = '<b>Status:</b> feedback salvo, mas ainda não há dados suficientes para re-treino.'
        return

    model = efficientnet
    model.train()
    transform = _hitl_transform()
    dataset = HITLFeedbackDataset(labeled_df, transform)
    loader = DataLoader(dataset, batch_size=HITL_RETRAIN_BATCH_SIZE, shuffle=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=HITL_RETRAIN_LR)

    for epoch in range(HITL_RETRAIN_EPOCHS):
        running_loss = 0.0
        for images_batch, labels_batch in loader:
            images_batch = images_batch.to(device)
            labels_batch = labels_batch.to(device)

            optimizer.zero_grad()
            outputs = model(images_batch)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images_batch.size(0)

        epoch_loss = running_loss / len(dataset)
        print(f'Epoch {epoch + 1}/{HITL_RETRAIN_EPOCHS} - loss: {epoch_loss:.4f}')

    checkpoint_candidates = [
        Path('best_efficientnet_finetuned.pth'),
        Path('best_efficientnet.pth'),
        Path('/content/drive/MyDrive/modelos/best_efficientnet_finetuned.pth'),
    ]
    checkpoint_path = next((path for path in checkpoint_candidates if path.exists()), checkpoint_candidates[0])
    torch.save(model.state_dict(), checkpoint_path)

    hitl_state['last_retrain_count'] = total_validated
    _save_state()
    if status_feedback is not None:
        status_feedback.value = f'<b>Status:</b> feedback salvo. Re-treino concluído e checkpoint atualizado em {checkpoint_path}.'

## 10. Interface HITL

Aqui você vai enviar a imagem, validar a predição e registrar o feedback do usuário.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

uploader = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False)
botao_analisar = widgets.Button(description='🩺 Analisar imagem', button_style='success', icon='search')
botao_correto = widgets.Button(description='✅ Correto', button_style='info', icon='check')
botao_incorreto = widgets.Button(description='❌ Incorreto', button_style='danger', icon='times')
botao_salvar_correcao = widgets.Button(description='Salvar correção', button_style='warning', icon='save')
dropdown_correcao = widgets.Dropdown(options=class_names, description='Classe correta:')
obs_input = widgets.Textarea(description='Obs.:', placeholder='Opcional', layout=widgets.Layout(width='100%', height='80px'))
saida_resultado = widgets.Output()
status_feedback = widgets.HTML()
correction_box = widgets.VBox([dropdown_correcao, obs_input, botao_salvar_correcao])
correction_box.layout.display = 'none'

current_sample = {
    'image': None,
    'predicted_label': None,
    'predicted_confidence': None,
    'raw_prediction': None,
    'raw_prediction_confidence': None,
    'probabilities': None,
}

print('=' * 72)
print('             DERMOSCAN HITL - FEEDBACK CONTINUOUS LEARNING')
print('=' * 72)
display(uploader)
display(botao_analisar)
display(saida_resultado)
display(status_feedback)
display(widgets.HBox([botao_correto, botao_incorreto]))
display(correction_box)


def _render_prediction(image_pil, predicted_label, predicted_confidence, probabilities):
    with saida_resultado:
        clear_output()
        plt.figure(figsize=(7, 7))
        plt.imshow(image_pil)
        plt.title(f'Predição final: {predicted_label} ({predicted_confidence:.2f}%)', fontsize=12)
        plt.axis('off')
        plt.tight_layout()
        plt.show()

        ranking = sorted(
            [(classe, probabilities[idx]) for idx, classe in label_map.items()],
            key=lambda item: item[1],
            reverse=True,
        )
        print('\nTop 3 classes:')
        for classe, prob in ranking[:3]:
            print(f'- {classe:<8} {prob * 100:.2f}%')


def processar_e_prever(_):
    if not uploader.value:
        with saida_resultado:
            clear_output()
            print('❌ Selecione uma imagem primeiro.')
        return

    content = _uploaded_content(uploader)
    image_pil = Image.open(io.BytesIO(content)).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    image_tensor = transform(image_pil).unsqueeze(0).to(device)

    efficientnet.eval()
    with torch.no_grad():
        outputs = efficientnet(image_tensor)
        probabilities = torch.softmax(outputs, dim=1).cpu().numpy()[0]

    raw_idx = int(np.argmax(probabilities))
    raw_label = label_map[raw_idx]
    raw_confidence = float(probabilities[raw_idx] * 100)

    classes_above_threshold = [idx for idx, classe in label_map.items() if probabilities[idx] >= dicionario_limiares[classe]]
    if len(classes_above_threshold) == 1:
        final_idx = classes_above_threshold[0]
    elif len(classes_above_threshold) > 1:
        final_idx = classes_above_threshold[int(np.argmax([probabilities[idx] for idx in classes_above_threshold]))]
    else:
        final_idx = raw_idx

    final_label = label_map[final_idx]
    final_confidence = float(probabilities[final_idx] * 100)

    current_sample.update({
        'image': image_pil.copy(),
        'predicted_label': final_label,
        'predicted_confidence': final_confidence,
        'raw_prediction': raw_label,
        'raw_prediction_confidence': raw_confidence,
        'probabilities': probabilities.copy(),
    })

    dropdown_correcao.value = final_label
    correction_box.layout.display = 'flex'
    status_feedback.value = '<b>Status:</b> validação aguardando feedback do usuário.'
    _render_prediction(image_pil, final_label, final_confidence, probabilities)


def registrar_feedback(feedback, correct_label=None):
    if current_sample['image'] is None:
        status_feedback.value = '<b>Status:</b> faça uma previsão antes de registrar feedback.'
        return

    if feedback == 'incorrect' and not correct_label:
        status_feedback.value = '<b>Status:</b> informe a classe correta antes de salvar a correção.'
        return

    final_correct_label = correct_label if correct_label is not None else current_sample['predicted_label']
    notes = obs_input.value.strip()

    _save_feedback(
        image_pil=current_sample['image'],
        predicted_label=current_sample['predicted_label'],
        predicted_confidence=current_sample['predicted_confidence'],
        raw_prediction=current_sample['raw_prediction'],
        raw_prediction_confidence=current_sample['raw_prediction_confidence'],
        feedback=feedback,
        correct_label=final_correct_label,
        notes=notes,
    )

    correction_box.layout.display = 'none'
    obs_input.value = ''
    status_feedback.value = f'<b>Status:</b> feedback salvo com sucesso. Total validado: {len(feedback_log)}.'
    _maybe_retrain()


def on_correto_clicked(_):
    registrar_feedback('correct')


def on_incorreto_clicked(_):
    correction_box.layout.display = 'flex'
    status_feedback.value = '<b>Status:</b> escolha a classe correta e clique em salvar correção.'


def on_salvar_correcao_clicked(_):
    registrar_feedback('incorrect', correct_label=dropdown_correcao.value)


botao_analisar.on_click(processar_e_prever)
botao_correto.on_click(on_correto_clicked)
botao_incorreto.on_click(on_incorreto_clicked)
botao_salvar_correcao.on_click(on_salvar_correcao_clicked)

if feedback_log.empty:
    status_feedback.value = '<b>Status:</b> ainda não há feedback armazenado.'
else:
    status_feedback.value = f'<b>Status:</b> {len(feedback_log)} feedbacks já armazenados.'